In [ ]:
!pip install -q transformers accelerate datasets peft bitsandbytes sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import ast
import gc
import json
import time
import math
import random
import warnings
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = Path("/content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser_2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = OUTPUT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ADAPTER_DIR = OUTPUT_DIR / "qwen2_5_1_5b_query_parser_lora"
METRICS_DIR = OUTPUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)


N_TRAIN = 8000
N_VALID = 1000
N_TEST = 1000

MAX_SEQ_LENGTH = 512

print("Model:", MODEL_NAME)
print("Output dir:", OUTPUT_DIR)

Model: Qwen/Qwen2.5-1.5B-Instruct
Output dir: /content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser_2


In [ ]:
JSON_SCHEMA_KEYS = [
    "city",
    "country",
    "price_bucket",
    "tags",
    "dietary",
    "matched_meals",
    "matched_features",
]

ALLOWED_PRICE_BUCKETS = ["cheap", "mid", "expensive"]

ALLOWED_DIETARY = [
    "vegetarian friendly",
    "vegan options",
    "gluten free options",
]

ALLOWED_MEALS = [
    "breakfast",
    "brunch",
    "lunch",
    "dinner",
    "drinks",
]

ALLOWED_FEATURES = [
    "outdoor seating",
    "reservations",
    "delivery",
    "wheelchair accessible",
    "family friendly",
    "romantic",
]

ALLOWED_TAGS = [
    "italian",
    "french",
    "spanish",
    "greek",
    "portuguese",
    "german",
    "seafood",
    "steakhouse",
    "asian",
    "indian",
    "mexican",
    "mediterranean",
    "fast food",
    "coffee",
    "bar",
    "fine dining",
    "tapas",
]

ALLOWED_CITIES = [
    "rome",
    "barcelona",
    "paris",
    "madrid",
    "milan",
    "lisbon",
    "athens",
    "porto",
    "florence",
    "venice",
    "naples",
    "valencia",
    "seville",
    "granada",
    "nice",
    "lyon",
    "marseille",
    "bordeaux",
    "vienna",
    "prague",
    "amsterdam",
    "brussels",
    "budapest",
]

ALLOWED_COUNTRIES = [
    "italy",
    "spain",
    "france",
    "portugal",
    "greece",
    "germany",
    "austria",
    "netherlands",
    "belgium",
    "hungary",
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PRICE_PHRASES = {
    "cheap": [
        "cheap",
        "budget",
        "affordable",
        "inexpensive",
        "low-cost",
        "not expensive",
        "somewhere that won't cost too much",
        "a place on a budget",
        "reasonably priced",
        "wallet-friendly",
    ],
    "mid": [
        "mid-range",
        "moderately priced",
        "not too cheap and not too expensive",
        "normal price",
        "average price",
        "casual mid-range",
        "medium-priced",
    ],
    "expensive": [
        "expensive",
        "fancy",
        "luxury",
        "premium",
        "high-end",
        "upscale",
        "fine dining",
        "special occasion",
        "elegant",
        "something more refined",
    ],
}

TAG_PHRASES = {
    "italian": ["Italian", "pizza", "pasta", "trattoria-style", "Italian food"],
    "french": ["French", "bistro", "brasserie", "French food"],
    "spanish": ["Spanish", "Spanish food", "paella"],
    "tapas": ["tapas", "tapas bar", "small plates"],
    "greek": ["Greek", "taverna-style", "Greek food"],
    "portuguese": ["Portuguese", "Portuguese food"],
    "german": ["German", "German food"],
    "seafood": ["seafood", "fish", "oysters", "seafood place"],
    "steakhouse": ["steakhouse", "steak", "grill"],
    "asian": ["Asian", "Chinese", "Japanese", "Thai", "Korean"],
    "indian": ["Indian", "curry", "Indian food"],
    "mexican": ["Mexican", "tacos", "burritos"],
    "mediterranean": ["Mediterranean", "Mediterranean food"],
    "fast food": ["fast food", "burger", "quick bite", "kebab"],
    "coffee": ["coffee", "cafe", "coffee shop", "café"],
    "bar": ["bar", "pub", "cocktails", "drinks"],
    "fine dining": ["fine dining", "gastronomic", "chef-driven"],
}

DIETARY_PHRASES = {
    "vegetarian friendly": [
        "vegetarian",
        "vegetarian-friendly",
        "veggie",
        "without meat",
        "meat-free",
    ],
    "vegan options": [
        "vegan",
        "plant-based",
        "vegan-friendly",
    ],
    "gluten free options": [
        "gluten-free",
        "without gluten",
        "gluten free",
    ],
}

MEAL_PHRASES = {
    "breakfast": ["breakfast", "morning meal", "early breakfast"],
    "brunch": ["brunch", "late breakfast", "weekend brunch"],
    "lunch": ["lunch", "midday meal"],
    "dinner": ["dinner", "evening meal", "tonight for dinner"],
    "drinks": ["drinks", "cocktails", "a drink", "beer or wine"],
}

FEATURE_PHRASES = {
    "outdoor seating": [
        "outdoor seating",
        "terrace",
        "sit outside",
        "outdoor table",
        "patio",
    ],
    "reservations": [
        "reservations",
        "book a table",
        "make a booking",
        "reserve a table",
    ],
    "delivery": [
        "delivery",
        "takeaway",
        "takeout",
        "food to go",
    ],
    "wheelchair accessible": [
        "wheelchair accessible",
        "accessible",
        "wheelchair access",
    ],
    "family friendly": [
        "family-friendly",
        "with kids",
        "for children",
        "family restaurant",
    ],
    "romantic": [
        "romantic",
        "date night",
        "for a date",
        "couples dinner",
    ],
}

In [ ]:
def normalize_text(x):
    if x is None:
        return ""

    s = str(x).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def make_empty_label():
    return {
        "city": None,
        "country": None,
        "price_bucket": None,
        "tags": [],
        "dietary": [],
        "matched_meals": [],
        "matched_features": [],
    }


def clean_label(label):
    clean = make_empty_label()

    city = label.get("city")
    country = label.get("country")
    price = label.get("price_bucket")

    clean["city"] = normalize_text(city) if city else None
    clean["country"] = normalize_text(country) if country else None

    if price in ALLOWED_PRICE_BUCKETS:
        clean["price_bucket"] = price
    else:
        clean["price_bucket"] = None

    for key, allowed in [
        ("tags", ALLOWED_TAGS),
        ("dietary", ALLOWED_DIETARY),
        ("matched_meals", ALLOWED_MEALS),
        ("matched_features", ALLOWED_FEATURES),
    ]:
        values = label.get(key, [])
        if values is None:
            values = []

        clean[key] = sorted(set([v for v in values if v in allowed]))

    return clean


def label_to_json(label):
    return json.dumps(clean_label(label), ensure_ascii=False, sort_keys=False)

In [ ]:
QUERY_TEMPLATES = [
    "Can you recommend me a {price_phrase} restaurant in {city}?",
    "I'm looking for a {price_phrase} place in {city}.",
    "Find me something {price_phrase} in {city}.",
    "Do you know any {price_phrase} restaurants in {city}?",
    "I need a {price_phrase} restaurant around {city}.",
    "Show me {price_phrase} food options in {city}.",
    "Where can I eat on a budget in {city}?",
    "I don't want to spend too much in {city}. Any suggestions?",
    "Suggest a reasonably priced restaurant in {city}.",
    "Give me a budget-friendly place in {city}.",

    "I want a {tag_phrase} place in {city}.",
    "Can you find a {tag_phrase} restaurant in {city}?",
    "Any good {tag_phrase} spots in {city}?",
    "Recommend a {tag_phrase} restaurant around {city}.",
    "I'm craving {tag_phrase} food in {city}.",
    "Where can I get {tag_phrase} in {city}?",
    "Find me a restaurant serving {tag_phrase} in {city}.",
    "I feel like eating {tag_phrase} in {city}.",

    "I need a {price_phrase} {tag_phrase} restaurant in {city}.",
    "Can you suggest a {price_phrase} {tag_phrase} place in {city}?",
    "Looking for {price_phrase} {tag_phrase} food in {city}.",
    "Find me a {tag_phrase} restaurant in {city} that is {price_phrase}.",
    "Any {price_phrase} {tag_phrase} restaurants near {city}?",
    "Recommend a {price_phrase} place for {tag_phrase} in {city}.",

    "Find me a {dietary_phrase} restaurant in {city}.",
    "I need a {dietary_phrase} place in {city}.",
    "Any {dietary_phrase} options in {city}?",
    "Can you recommend a restaurant with {dietary_phrase} options in {city}?",
    "Looking for {dietary_phrase} food in {city}.",
    "I have dietary restrictions. I need {dietary_phrase} options in {city}.",

    "Find me a {dietary_phrase} {tag_phrase} restaurant in {city}.",
    "I want a {dietary_phrase} {tag_phrase} place in {city}.",
    "Can you recommend a {tag_phrase} restaurant with {dietary_phrase} options in {city}?",
    "Any {dietary_phrase} {tag_phrase} spots in {city}?",
    "Looking for {tag_phrase} food in {city}, but it needs to be {dietary_phrase}.",

    "Any good places for {meal_phrase} in {city}?",
    "I need a spot for {meal_phrase} in {city}.",
    "Where should I go for {meal_phrase} in {city}?",
    "Can you suggest somewhere for {meal_phrase} in {city}?",
    "Find me a restaurant for {meal_phrase} in {city}.",

    "Any {tag_phrase} spots for {meal_phrase} in {city}?",
    "I want {tag_phrase} food for {meal_phrase} in {city}.",
    "Find me a {tag_phrase} place for {meal_phrase} in {city}.",
    "Recommend a {tag_phrase} restaurant for {meal_phrase} in {city}.",

    "Looking for a restaurant with {feature_phrase} in {city}.",
    "Find me a place with {feature_phrase} in {city}.",
    "Can you suggest a restaurant in {city} that has {feature_phrase}?",
    "I need a restaurant with {feature_phrase} around {city}.",
    "Any spots with {feature_phrase} in {city}?",

    "Looking for a {price_phrase} restaurant with {feature_phrase} in {city}.",
    "Find me a {price_phrase} place in {city} with {feature_phrase}.",
    "Can you suggest somewhere {price_phrase} in {city} that has {feature_phrase}?",

    "I need a {feature_phrase} place for {meal_phrase} in {city}.",
    "Find me somewhere for {meal_phrase} with {feature_phrase} in {city}.",
    "Can you recommend a restaurant with {feature_phrase} for {meal_phrase} in {city}?",

    "Can you recommend a {price_phrase} {dietary_phrase} {tag_phrase} place in {city}?",
    "I want a {price_phrase} {tag_phrase} restaurant in {city} with {feature_phrase}.",
    "Find me a {dietary_phrase} place for {meal_phrase} in {city}.",
    "I need a {price_phrase} restaurant for {meal_phrase} in {city} with {feature_phrase}.",
    "Any {price_phrase} {tag_phrase} spots for {meal_phrase} in {city}?",
    "I'm looking for a {dietary_phrase} {tag_phrase} restaurant for {meal_phrase} in {city}.",
    "Can you find a {price_phrase} {dietary_phrase} restaurant with {feature_phrase} in {city}?",
    "Suggest a {tag_phrase} place in {city} for {meal_phrase} that is {price_phrase}.",

    "I'm visiting {city} and want something {price_phrase}.",
    "I'll be in {city}; where can I find {tag_phrase} food?",
    "I'm traveling to {city}. I need a {dietary_phrase} restaurant.",
    "Going out in {city} tonight. I want {tag_phrase} for {meal_phrase}.",
    "I'm in {city} and need a place with {feature_phrase}.",
    "Could you help me find a {price_phrase} restaurant in {city}?",
    "What would you suggest in {city} for {meal_phrase}?",
    "I have a date in {city}. Find me somewhere romantic for dinner.",
    "We are going with kids in {city}. Find a family-friendly place.",
    "I need somewhere accessible in {city}.",
    "I want to sit outside in {city}.",
    "I need to book a table in {city}.",
    "Something fancy in {city} for dinner, please.",
    "Something casual and not too expensive in {city}.",
    "I want a quick bite in {city}.",
    "Where can I get coffee and breakfast in {city}?",
    "I'm looking for tapas in {city}.",
    "I want seafood near {city}, preferably not too expensive.",
    "Find gluten-free dinner options in {city}.",
    "Can you recommend a plant-based brunch place in {city}?",
]

In [ ]:
def pick_from_dict(d):
    canonical = random.choice(list(d.keys()))
    phrase = random.choice(d[canonical])
    return canonical, phrase


def fill_template(template):
    label = make_empty_label()

    city = random.choice(ALLOWED_CITIES)
    label["city"] = city

    values = {
        "city": city.title(),
    }

    if "{price_phrase}" in template:
        price, price_phrase = pick_from_dict(PRICE_PHRASES)
        label["price_bucket"] = price
        values["price_phrase"] = price_phrase

    if "{tag_phrase}" in template:
        tag, tag_phrase = pick_from_dict(TAG_PHRASES)
        label["tags"].append(tag)
        values["tag_phrase"] = tag_phrase

    if "{dietary_phrase}" in template:
        dietary, dietary_phrase = pick_from_dict(DIETARY_PHRASES)
        label["dietary"].append(dietary)
        values["dietary_phrase"] = dietary_phrase

    if "{meal_phrase}" in template:
        meal, meal_phrase = pick_from_dict(MEAL_PHRASES)
        label["matched_meals"].append(meal)
        values["meal_phrase"] = meal_phrase

    if "{feature_phrase}" in template:
        feature, feature_phrase = pick_from_dict(FEATURE_PHRASES)
        label["matched_features"].append(feature)
        values["feature_phrase"] = feature_phrase

    t_norm = template.lower()

    if "date" in t_norm or "romantic" in t_norm:
        if "romantic" not in label["matched_features"]:
            label["matched_features"].append("romantic")

    if "kids" in t_norm or "family-friendly" in t_norm or "family friendly" in t_norm:
        if "family friendly" not in label["matched_features"]:
            label["matched_features"].append("family friendly")

    if "accessible" in t_norm:
        if "wheelchair accessible" not in label["matched_features"]:
            label["matched_features"].append("wheelchair accessible")

    if "sit outside" in t_norm:
        if "outdoor seating" not in label["matched_features"]:
            label["matched_features"].append("outdoor seating")

    if "book a table" in t_norm:
        if "reservations" not in label["matched_features"]:
            label["matched_features"].append("reservations")

    if "fancy" in t_norm:
        label["price_bucket"] = "expensive"

    if "not too expensive" in t_norm:
        label["price_bucket"] = "mid"

    if "quick bite" in t_norm:
        if "fast food" not in label["tags"]:
            label["tags"].append("fast food")

    if "coffee and breakfast" in t_norm:
        if "coffee" not in label["tags"]:
            label["tags"].append("coffee")
        if "breakfast" not in label["matched_meals"]:
            label["matched_meals"].append("breakfast")

    if "tapas" in t_norm:
        if "tapas" not in label["tags"]:
            label["tags"].append("tapas")

    if "seafood" in t_norm:
        if "seafood" not in label["tags"]:
            label["tags"].append("seafood")

    if "gluten-free" in t_norm or "gluten free" in t_norm:
        if "gluten free options" not in label["dietary"]:
            label["dietary"].append("gluten free options")

    if "plant-based" in t_norm:
        if "vegan options" not in label["dietary"]:
            label["dietary"].append("vegan options")

    if "brunch" in t_norm:
        if "brunch" not in label["matched_meals"]:
            label["matched_meals"].append("brunch")

    if "dinner" in t_norm or "tonight" in t_norm:
        if "dinner" not in label["matched_meals"]:
            label["matched_meals"].append("dinner")

    query = template.format(**values)
    label = clean_label(label)

    return query, label


def generate_synthetic_examples(n_examples):
    rows = []

    seen = set()
    attempts = 0

    while len(rows) < n_examples and attempts < n_examples * 20:
        attempts += 1

        template = random.choice(QUERY_TEMPLATES)
        query, label = fill_template(template)

        key = (query, label_to_json(label))

        if key in seen:
            continue

        seen.add(key)

        rows.append({
            "query": query,
            "label": label,
            "label_json": label_to_json(label),
            "template": template,
        })

    return pd.DataFrame(rows)


total_examples = N_TRAIN + N_VALID + N_TEST
synthetic_df = generate_synthetic_examples(total_examples)

print("Generated examples:", len(synthetic_df))
display(synthetic_df.head(10))

Generated examples: 10000


,query,label,label_json,template
0,Find gluten-free dinner options in Madrid.,"{'city': 'madrid', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': ['gluten free options'], 'matched_meals': ['dinner'], 'matched_features': []}","{""city"": ""madrid"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [""dinner""], ""matched_features"": []}",Find gluten-free dinner options in {city}.
1,Do you know any inexpensive restaurants in Florence?,"{'city': 'florence', 'country': None, 'price_bucket': 'cheap', 'tags': [], 'dietary': [], 'matched_meals': [], 'matched_features': []}","{""city"": ""florence"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",Do you know any {price_phrase} restaurants in {city}?
2,I feel like eating paella in Madrid.,"{'city': 'madrid', 'country': None, 'price_bucket': None, 'tags': ['spanish'], 'dietary': [], 'matched_meals': [], 'matched_features': []}","{""city"": ""madrid"", ""country"": null, ""price_bucket"": null, ""tags"": [""spanish""], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",I feel like eating {tag_phrase} in {city}.
3,Can you recommend a restaurant with book a table for breakfast in Barcelona?,"{'city': 'barcelona', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': [], 'matched_meals': ['breakfast'], 'matched_features': ['reservations']}","{""city"": ""barcelona"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [], ""matched_meals"": [""breakfast""], ""matched_features"": [""reservations""]}",Can you recommend a restaurant with {feature_phrase} for {meal_phrase} in {city}?
4,I'll be in Prague; where can I find Italian food food?,"{'city': 'prague', 'country': None, 'price_bucket': None, 'tags': ['italian'], 'dietary': [], 'matched_meals': [], 'matched_features': []}","{""city"": ""prague"", ""country"": null, ""price_bucket"": null, ""tags"": [""italian""], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",I'll be in {city}; where can I find {tag_phrase} food?
5,I need a gluten free place in Budapest.,"{'city': 'budapest', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': ['gluten free options'], 'matched_meals': [], 'matched_features': []}","{""city"": ""budapest"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""gluten free options""], ""matched_meals"": [], ""matched_features"": []}",I need a {dietary_phrase} place in {city}.
6,What would you suggest in Granada for late breakfast?,"{'city': 'granada', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': [], 'matched_meals': ['brunch'], 'matched_features': []}","{""city"": ""granada"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [], ""matched_meals"": [""brunch""], ""matched_features"": []}",What would you suggest in {city} for {meal_phrase}?
7,"Something fancy in Florence for dinner, please.","{'city': 'florence', 'country': None, 'price_bucket': 'expensive', 'tags': [], 'dietary': [], 'matched_meals': ['dinner'], 'matched_features': []}","{""city"": ""florence"", ""country"": null, ""price_bucket"": ""expensive"", ""tags"": [], ""dietary"": [], ""matched_meals"": [""dinner""], ""matched_features"": []}","Something fancy in {city} for dinner, please."
8,Can you recommend me a fine dining restaurant in Lisbon?,"{'city': 'lisbon', 'country': None, 'price_bucket': 'expensive', 'tags': [], 'dietary': [], 'matched_meals': [], 'matched_features': []}","{""city"": ""lisbon"", ""country"": null, ""price_bucket"": ""expensive"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",Can you recommend me a {price_phrase} restaurant in {city}?
9,Recommend a Greek restaurant for lunch in Florence.,"{'city': 'florence', 'country': None, 'price_bucket': None, 'tags': ['greek'], 'dietary': [], 'matched_meals': ['lunch'], 'matched_f

In [ ]:
manual_examples = [
    (
        "Can you recommend me a cheap restaurant in Barcelona?",
        {
            "city": "barcelona",
            "country": None,
            "price_bucket": "cheap",
            "tags": [],
            "dietary": [],
            "matched_meals": [],
            "matched_features": [],
        },
    ),
    (
        "Somewhere nice in Paris for a date night, not too expensive.",
        {
            "city": "paris",
            "country": None,
            "price_bucket": "mid",
            "tags": [],
            "dietary": [],
            "matched_meals": ["dinner"],
            "matched_features": ["romantic"],
        },
    ),
    (
        "I want a fancy gluten-free place for dinner in Milan.",
        {
            "city": "milan",
            "country": None,
            "price_bucket": "expensive",
            "tags": [],
            "dietary": ["gluten free options"],
            "matched_meals": ["dinner"],
            "matched_features": [],
        },
    ),
    (
        "Do you know any affordable seafood restaurants in Lisbon with outdoor seating?",
        {
            "city": "lisbon",
            "country": None,
            "price_bucket": "cheap",
            "tags": ["seafood"],
            "dietary": [],
            "matched_meals": [],
            "matched_features": ["outdoor seating"],
        },
    ),
    (
        "Find me a vegan brunch place in Rome.",
        {
            "city": "rome",
            "country": None,
            "price_bucket": None,
            "tags": [],
            "dietary": ["vegan options"],
            "matched_meals": ["brunch"],
            "matched_features": [],
        },
    ),
    (
        "I need coffee and breakfast in Athens.",
        {
            "city": "athens",
            "country": None,
            "price_bucket": None,
            "tags": ["coffee"],
            "dietary": [],
            "matched_meals": ["breakfast"],
            "matched_features": [],
        },
    ),
    (
        "Can you find tapas in Madrid?",
        {
            "city": "madrid",
            "country": None,
            "price_bucket": None,
            "tags": ["tapas"],
            "dietary": [],
            "matched_meals": [],
            "matched_features": [],
        },
    ),
    (
        "I need a family-friendly place in Valencia.",
        {
            "city": "valencia",
            "country": None,
            "price_bucket": None,
            "tags": [],
            "dietary": [],
            "matched_meals": [],
            "matched_features": ["family friendly"],
        },
    ),
]

manual_df = pd.DataFrame([
    {
        "query": q,
        "label": clean_label(label),
        "label_json": label_to_json(label),
        "template": "manual",
    }
    for q, label in manual_examples
])

synthetic_df = pd.concat([manual_df, synthetic_df], ignore_index=True)
synthetic_df = synthetic_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Total after manual examples:", len(synthetic_df))
display(synthetic_df.head(10))

Total after manual examples: 10008


,query,label,label_json,template
0,I feel like eating steak in Budapest.,"{'city': 'budapest', 'country': None, 'price_bucket': None, 'tags': ['steakhouse'], 'dietary': [], 'matched_meals': [], 'matched_features': []}","{""city"": ""budapest"", ""country"": null, ""price_bucket"": null, ""tags"": [""steakhouse""], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",I feel like eating {tag_phrase} in {city}.
1,Can you recommend a restaurant with veggie options in Brussels?,"{'city': 'brussels', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': ['vegetarian friendly'], 'matched_meals': [], 'matched_features': []}","{""city"": ""brussels"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [""vegetarian friendly""], ""matched_meals"": [], ""matched_features"": []}",Can you recommend a restaurant with {dietary_phrase} options in {city}?
2,I need a reserve a table place for midday meal in Milan.,"{'city': 'milan', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': [], 'matched_meals': ['lunch'], 'matched_features': ['reservations']}","{""city"": ""milan"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [], ""matched_meals"": [""lunch""], ""matched_features"": [""reservations""]}",I need a {feature_phrase} place for {meal_phrase} in {city}.
3,Find me a high-end place in Venice with takeout.,"{'city': 'venice', 'country': None, 'price_bucket': 'expensive', 'tags': [], 'dietary': [], 'matched_meals': [], 'matched_features': ['delivery']}","{""city"": ""venice"", ""country"": null, ""price_bucket"": ""expensive"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""delivery""]}",Find me a {price_phrase} place in {city} with {feature_phrase}.
4,"Looking for quick bite food in Marseille, but it needs to be vegetarian.","{'city': 'marseille', 'country': None, 'price_bucket': None, 'tags': ['fast food'], 'dietary': ['vegetarian friendly'], 'matched_meals': [], 'matched_features': []}","{""city"": ""marseille"", ""country"": null, ""price_bucket"": null, ""tags"": [""fast food""], ""dietary"": [""vegetarian friendly""], ""matched_meals"": [], ""matched_features"": []}","Looking for {tag_phrase} food in {city}, but it needs to be {dietary_phrase}."
5,Where should I go for drinks in Prague?,"{'city': 'prague', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': [], 'matched_meals': ['drinks'], 'matched_features': []}","{""city"": ""prague"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [], ""matched_meals"": [""drinks""], ""matched_features"": []}",Where should I go for {meal_phrase} in {city}?
6,Where can I get steak in Granada?,"{'city': 'granada', 'country': None, 'price_bucket': None, 'tags': ['steakhouse'], 'dietary': [], 'matched_meals': [], 'matched_features': []}","{""city"": ""granada"", ""country"": null, ""price_bucket"": null, ""tags"": [""steakhouse""], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",Where can I get {tag_phrase} in {city}?
7,Find me a restaurant for dinner in Venice.,"{'city': 'venice', 'country': None, 'price_bucket': None, 'tags': [], 'dietary': [], 'matched_meals': ['dinner'], 'matched_features': []}","{""city"": ""venice"", ""country"": null, ""price_bucket"": null, ""tags"": [], ""dietary"": [], ""matched_meals"": [""dinner""], ""matched_features"": []}",Find me a restaurant for {meal_phrase} in {city}.
8,Find me a affordable place in Venice with reservations.,"{'city': 'venice', 'country': None, 'price_bucket': 'cheap', 'tags': [], 'dietary': [], 'matched_meals': [], 'matched_features': ['reservations']}","{""city"": ""venice"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""reservations""]}",Find me a {price_phrase} place in {city} with {feature_phrase}.
9,Can you find a something more refined without gluten restaurant with with kids in Lisbon?,"{'city': 'lisbon', 'cou

In [ ]:
train_df = synthetic_df.iloc[:N_TRAIN].copy().reset_index(drop=True)
valid_df = synthetic_df.iloc[N_TRAIN:N_TRAIN + N_VALID].copy().reset_index(drop=True)
test_df = synthetic_df.iloc[N_TRAIN + N_VALID:N_TRAIN + N_VALID + N_TEST].copy().reset_index(drop=True)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test:", test_df.shape)

train_path = DATA_DIR / "slm_parser_train.csv"
valid_path = DATA_DIR / "slm_parser_valid.csv"
test_path = DATA_DIR / "slm_parser_test.csv"

train_df.to_csv(train_path, index=False)
valid_df.to_csv(valid_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved:")
print(train_path)
print(valid_path)
print(test_path)

Train: (8000, 4)
Valid: (1000, 4)
Test: (1000, 4)
Saved:
/content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser/data/slm_parser_train.csv
/content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser/data/slm_parser_valid.csv
/content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser/data/slm_parser_test.csv


In [ ]:
SYSTEM_PROMPT = """You are a query parser for a restaurant recommendation system.
Extract structured restaurant search constraints from the user query.
Return valid JSON only. Do not explain anything."""

USER_PROMPT_TEMPLATE = """Return valid JSON only with this schema:
{{
  "city": string or null,
  "country": string or null,
  "price_bucket": "cheap" or "mid" or "expensive" or null,
  "tags": list of strings,
  "dietary": list of strings,
  "matched_meals": list of strings,
  "matched_features": list of strings
}}

Allowed values:
price_bucket: cheap, mid, expensive
tags: italian, french, spanish, greek, portuguese, german, seafood, steakhouse, asian, indian, mexican, mediterranean, fast food, coffee, bar, fine dining, tapas
dietary: vegetarian friendly, vegan options, gluten free options
matched_meals: breakfast, brunch, lunch, dinner, drinks
matched_features: outdoor seating, reservations, delivery, wheelchair accessible, family friendly, romantic

User query:
{query}

JSON:"""


def build_messages(query, label_json=None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(query=query)},
    ]

    if label_json is not None:
        messages.append({"role": "assistant", "content": label_json})

    return messages

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [ ]:
def build_prompt_text(query):
    messages = build_messages(query=query, label_json=None)

    if hasattr(tokenizer, "apply_chat_template"):
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt_text = (
            f"SYSTEM: {messages[0]['content']}\n\n"
            f"USER: {messages[1]['content']}\n\n"
            f"ASSISTANT:"
        )

    return prompt_text


def build_full_text_and_prompt_len(row):
    prompt_text = build_prompt_text(row["query"])
    answer_text = row["label_json"] + tokenizer.eos_token

    full_text = prompt_text + answer_text

    prompt_ids = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )["input_ids"]

    return full_text, len(prompt_ids)


train_df[["text", "prompt_len"]] = train_df.apply(
    lambda row: pd.Series(build_full_text_and_prompt_len(row)),
    axis=1,
)

valid_df[["text", "prompt_len"]] = valid_df.apply(
    lambda row: pd.Series(build_full_text_and_prompt_len(row)),
    axis=1,
)

print(train_df[["query", "label_json", "prompt_len"]].head())
print("\nTraining text sample:\n")
print(train_df["text"].iloc[0][:2000])

                                                                      query  \
0                                     I feel like eating steak in Budapest.   
1           Can you recommend a restaurant with veggie options in Brussels?   
2                  I need a reserve a table place for midday meal in Milan.   
3                          Find me a high-end place in Venice with takeout.   
4  Looking for quick bite food in Marseille, but it needs to be vegetarian.   

                                                                                                                                                             label_json  prompt_len  
0                       {"city": "budapest", "country": null, "price_bucket": null, "tags": ["steakhouse"], "dietary": [], "matched_meals": [], "matched_features": []}         242  
1              {"city": "brussels", "country": null, "price_bucket": null, "tags": [], "dietary": ["vegetarian friendly"], "matched_meals": [], "matched_features

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["text"]])
valid_dataset = Dataset.from_pandas(valid_df[["text"]])
test_dataset = Dataset.from_pandas(test_df[["query", "label_json", "label"]])

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1000
    })
})

In [ ]:
def tokenize_with_masked_prompt(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

    labels = []

    for input_ids, prompt_len in zip(tokenized["input_ids"], examples["prompt_len"]):
        label_ids = input_ids.copy()

        mask_until = min(int(prompt_len), len(label_ids))
        label_ids[:mask_until] = [-100] * mask_until

        labels.append(label_ids)

    tokenized["labels"] = labels

    return tokenized


train_dataset = Dataset.from_pandas(train_df[["text", "prompt_len"]])
valid_dataset = Dataset.from_pandas(valid_df[["text", "prompt_len"]])

tokenized_train = train_dataset.map(
    tokenize_with_masked_prompt,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_valid = valid_dataset.map(
    tokenize_with_masked_prompt,
    batched=True,
    remove_columns=valid_dataset.column_names,
)

print(tokenized_train)
print(tokenized_valid)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

model.config.use_cache = False

print("Model loaded.")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [ ]:
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    print('yes')
if torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
    print('yee')

yes


In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
print(OUTPUT_DIR)

/content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser_2


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List


@dataclass
class DataCollatorForCausalLMWithMaskedPrompt:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        labels = [f["labels"] for f in features]

        batch = self.tokenizer.pad(
            {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            },
            padding=True,
            return_tensors="pt",
        )

        max_len = batch["input_ids"].shape[1]

        padded_labels = []

        for label in labels:
            pad_len = max_len - len(label)
            padded_labels.append(label + [-100] * pad_len)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        return batch


data_collator = DataCollatorForCausalLMWithMaskedPrompt(tokenizer=tokenizer)

print("Custom masked-prompt data collator ready.")

Custom masked-prompt data collator ready.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

print("Trainer ready.")

Trainer ready.


In [ ]:
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time

print(f"Training time: {elapsed / 60:.2f} minutes")
train_result

Step,Training Loss,Validation Loss
250,0.000804,0.001164
500,0.000287,0.000559


Training time: 14.75 minutes


TrainOutput(global_step=500, training_loss=0.013733870238997041, metrics={'train_runtime': 884.3756, 'train_samples_per_second': 18.092, 'train_steps_per_second': 0.565, 'total_flos': 3.814049872891085e+16, 'train_loss': 0.013733870238997041, 'epoch': 2.0})

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved LoRA adapter and tokenizer to:", ADAPTER_DIR)

Saved LoRA adapter and tokenizer to: /content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser_2/qwen2_5_1_5b_query_parser_lora


In [ ]:
def build_inference_prompt(query):
    messages = build_messages(query=query, label_json=None)

    if hasattr(tokenizer, "apply_chat_template"):
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt_text = (
            f"SYSTEM: {messages[0]['content']}\n\n"
            f"USER: {messages[1]['content']}\n\n"
            f"ASSISTANT:"
        )

    return prompt_text


def extract_json_from_text(text):
    if text is None:
        return None

    s = str(text).strip()


    try:
        return json.loads(s)
    except Exception:
        pass


    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if m:
        candidate = m.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            return None

    return None


def validate_and_repair_parsed_json(obj):
    if not isinstance(obj, dict):
        return None

    repaired = make_empty_label()

    city = obj.get("city")
    country = obj.get("country")
    price = obj.get("price_bucket")

    city_norm = normalize_text(city) if city else None
    country_norm = normalize_text(country) if country else None

    repaired["city"] = city_norm if city_norm else None
    repaired["country"] = country_norm if country_norm else None
    repaired["price_bucket"] = price if price in ALLOWED_PRICE_BUCKETS else None

    for key, allowed in [
        ("tags", ALLOWED_TAGS),
        ("dietary", ALLOWED_DIETARY),
        ("matched_meals", ALLOWED_MEALS),
        ("matched_features", ALLOWED_FEATURES),
    ]:
        values = obj.get(key, [])

        if values is None:
            values = []

        if isinstance(values, str):
            values = [values]

        if not isinstance(values, list):
            values = []

        cleaned_values = []

        for v in values:
            v_norm = normalize_text(v)
            if v_norm in allowed:
                cleaned_values.append(v_norm)

        repaired[key] = sorted(set(cleaned_values))

    return repaired


def slm_parse_query(query, max_new_tokens=180):
    prompt_text = build_inference_prompt(query)

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    new_tokens = generated[0][input_len:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    parsed_obj = extract_json_from_text(raw_output)
    repaired = validate_and_repair_parsed_json(parsed_obj)

    return {
        "query": query,
        "raw_output": raw_output,
        "parsed_json": parsed_obj,
        "validated_json": repaired,
        "valid_json": repaired is not None,
    }

In [ ]:
manual_test_queries = [
    "Can you recommend me a cheap restaurant in Barcelona?",
    "I want a fancy gluten-free place for dinner in Milan.",
    "Do you know any affordable seafood restaurants in Lisbon with outdoor seating?",
    "Find me a vegan brunch place in Rome.",
    "Somewhere nice in Paris for a date night, not too expensive.",
    "I need coffee and breakfast in Athens.",
    "Can you find tapas in Madrid?",
]

for q in manual_test_queries:
    out = slm_parse_query(q)

    print("=" * 120)
    print("QUERY:", q)
    print("RAW:")
    print(out["raw_output"])
    print("VALIDATED:")
    print(json.dumps(out["validated_json"], indent=2, ensure_ascii=False))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUERY: Can you recommend me a cheap restaurant in Barcelona?
RAW:
{"city": "barcelona", "country": null, "price_bucket": "cheap", "tags": [], "dietary": [], "matched_meals": [], "matched_features": []}
VALIDATED:
{
  "city": "barcelona",
  "country": null,
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
QUERY: I want a fancy gluten-free place for dinner in Milan.
RAW:
{"city": "milan", "country": null, "price_bucket": "expensive", "tags": [], "dietary": ["gluten free options"], "matched_meals": ["dinner"], "matched_features": []}
VALIDATED:
{
  "city": "milan",
  "country": null,
  "price_bucket": "expensive",
  "tags": [],
  "dietary": [
    "gluten free options"
  ],
  "matched_meals": [
    "dinner"
  ],
  "matched_features": []
}
QUERY: Do you know any affordable seafood restaurants in Lisbon with outdoor seating?
RAW:
{"city": "lisbon", "country": null, "price_bucket": "cheap", "tags": ["seafood"], "dietary": [], "matched

In [ ]:
def parse_label_json(label_json):
    try:
        return clean_label(json.loads(label_json))
    except Exception:
        return make_empty_label()


def exact_match(pred, gold):
    return pred == gold


def scalar_accuracy(preds, golds, key):
    correct = 0
    total = 0

    for p, g in zip(preds, golds):
        correct += int(p.get(key) == g.get(key))
        total += 1

    return correct / total if total else None


def list_precision_recall_f1(pred_values, gold_values):
    pred_set = set(pred_values or [])
    gold_set = set(gold_values or [])

    if len(pred_set) == 0 and len(gold_set) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_set) == 0:
        return 0.0, 0.0, 0.0

    if len(gold_set) == 0:
        return 0.0, 0.0, 0.0

    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(gold_set) if gold_set else 0.0

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


def evaluate_slm_parser(test_dataframe, max_eval_examples=100):
    eval_df = test_dataframe.head(max_eval_examples).copy()

    predictions = []
    golds = []
    rows = []

    for i, row in eval_df.iterrows():
        query = row["query"]
        gold = parse_label_json(row["label_json"])

        out = slm_parse_query(query)
        pred = out["validated_json"] if out["validated_json"] is not None else make_empty_label()

        predictions.append(pred)
        golds.append(gold)

        rows.append({
            "query": query,
            "gold_json": json.dumps(gold, ensure_ascii=False),
            "raw_output": out["raw_output"],
            "pred_json": json.dumps(pred, ensure_ascii=False),
            "valid_json": out["valid_json"],
            "exact_match": exact_match(pred, gold),
        })

    results_df = pd.DataFrame(rows)

    metrics = {
        "num_examples": len(eval_df),
        "valid_json_rate": results_df["valid_json"].mean(),
        "exact_match_rate": results_df["exact_match"].mean(),
        "city_accuracy": scalar_accuracy(predictions, golds, "city"),
        "country_accuracy": scalar_accuracy(predictions, golds, "country"),
        "price_bucket_accuracy": scalar_accuracy(predictions, golds, "price_bucket"),
    }

    for key in ["tags", "dietary", "matched_meals", "matched_features"]:
        ps = []
        rs = []
        fs = []

        for p, g in zip(predictions, golds):
            precision, recall, f1 = list_precision_recall_f1(p.get(key, []), g.get(key, []))
            ps.append(precision)
            rs.append(recall)
            fs.append(f1)

        metrics[f"{key}_precision"] = float(np.mean(ps))
        metrics[f"{key}_recall"] = float(np.mean(rs))
        metrics[f"{key}_f1"] = float(np.mean(fs))

    return results_df, pd.DataFrame([metrics])


eval_results_df, eval_metrics_df = evaluate_slm_parser(test_df, max_eval_examples=100)

display(eval_metrics_df)
display(eval_results_df.head(20))

,num_examples,valid_json_rate,exact_match_rate,city_accuracy,country_accuracy,price_bucket_accuracy,tags_precision,tags_recall,tags_f1,dietary_precision,dietary_recall,dietary_f1,matched_meals_precision,matched_meals_recall,matched_meals_f1,matched_features_precision,matched_features_recall,matched_features_f1
0,100,1.0,0.99,1.0,1.0,1.0,0.99,0.99,0.99,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


,query,gold_json,raw_output,pred_json,valid_json,exact_match
0,Where can I get brasserie in Prague?,"{""city"": ""prague"", ""country"": null, ""price_bucket"": null, ""tags"": [""french""], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}","{""city"": ""prague"", ""country"": null, ""price_bucket"": null, ""tags"": [""french""], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}","{""city"": ""prague"", ""country"": null, ""price_bucket"": null, ""tags"": [""french""], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",True,True
1,Can you suggest somewhere mid-range in Bordeaux that has food to go?,"{""city"": ""bordeaux"", ""country"": null, ""price_bucket"": ""mid"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""delivery""]}","{""city"": ""bordeaux"", ""country"": null, ""price_bucket"": ""mid"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""delivery""]}","{""city"": ""bordeaux"", ""country"": null, ""price_bucket"": ""mid"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""delivery""]}",True,True
2,Any veggie Spanish food spots in Amsterdam?,"{""city"": ""amsterdam"", ""country"": null, ""price_bucket"": null, ""tags"": [""spanish""], ""dietary"": [""vegetarian friendly""], ""matched_meals"": [], ""matched_features"": []}","{""city"": ""amsterdam"", ""country"": null, ""price_bucket"": null, ""tags"": [""spanish""], ""dietary"": [""vegetarian friendly""], ""matched_meals"": [], ""matched_features"": []}","{""city"": ""amsterdam"", ""country"": null, ""price_bucket"": null, ""tags"": [""spanish""], ""dietary"": [""vegetarian friendly""], ""matched_meals"": [], ""matched_features"": []}",True,True
3,Looking for a cheap restaurant with reserve a table in Nice.,"{""city"": ""nice"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""reservations""]}","{""city"": ""nice"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""reservations""]}","{""city"": ""nice"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""reservations""]}",True,True
4,Looking for a normal price restaurant with family-friendly in Vienna.,"{""city"": ""vienna"", ""country"": null, ""price_bucket"": ""mid"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""family friendly""]}","{""city"": ""vienna"", ""country"": null, ""price_bucket"": ""mid"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""family friendly""]}","{""city"": ""vienna"", ""country"": null, ""price_bucket"": ""mid"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""family friendly""]}",True,True
5,I'm visiting Madrid and want something upscale.,"{""city"": ""madrid"", ""country"": null, ""price_bucket"": ""expensive"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}","{""city"": ""madrid"", ""country"": null, ""price_bucket"": ""expensive"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}","{""city"": ""madrid"", ""country"": null, ""price_bucket"": ""expensive"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": []}",True,True
6,Find me a low-cost place in Milan with patio.,"{""city"": ""milan"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""outdoor seating""]}","{""city"": ""milan"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_features"": [""outdoor seating""]}","{""city"": ""milan"", ""country"": null, ""price_bucket"": ""cheap"", ""tags"": [], ""dietary"": [], ""matched_meals"": [], ""matched_feature

In [ ]:
eval_results_path = METRICS_DIR / "slm_parser_eval_results.csv"
eval_metrics_path = METRICS_DIR / "slm_parser_eval_metrics.csv"

eval_results_df.to_csv(eval_results_path, index=False)
eval_metrics_df.to_csv(eval_metrics_path, index=False)

print("Saved:")
print(eval_results_path)
print(eval_metrics_path)

Saved:
/content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser_2/metrics/slm_parser_eval_results.csv
/content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser_2/metrics/slm_parser_eval_metrics.csv


In [ ]:
demo_query = "Can you recommend me a cheap vegetarian Italian restaurant in Rome?"

slm_output = slm_parse_query(demo_query)

print("User query:")
print(demo_query)

print("\nSLM raw output:")
print(slm_output["raw_output"])

print("\nValidated JSON constraints:")
print(json.dumps(slm_output["validated_json"], indent=2, ensure_ascii=False))

print("\nNext pipeline step:")
print("validated_json -> metadata filtering -> FAISS retrieval -> reranking -> RAG response generation")

User query:
Can you recommend me a cheap vegetarian Italian restaurant in Rome?

SLM raw output:
{"city": "rome", "country": null, "price_bucket": "cheap", "tags": ["italian"], "dietary": ["vegetarian friendly"], "matched_meals": [], "matched_features": []}

Validated JSON constraints:
{
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Next pipeline step:
validated_json -> metadata filtering -> FAISS retrieval -> reranking -> RAG response generation


In [ ]:
slm_config = {
    "base_model": MODEL_NAME,
    "task": "restaurant_query_parsing_to_json",
    "training_method": "QLoRA / LoRA adapter fine-tuning",
    "num_train_examples": int(len(train_df)),
    "num_validation_examples": int(len(valid_df)),
    "num_test_examples": int(len(test_df)),
    "max_seq_length": MAX_SEQ_LENGTH,
    "json_schema_keys": JSON_SCHEMA_KEYS,
    "allowed_price_buckets": ALLOWED_PRICE_BUCKETS,
    "allowed_tags": ALLOWED_TAGS,
    "allowed_dietary": ALLOWED_DIETARY,
    "allowed_meals": ALLOWED_MEALS,
    "allowed_features": ALLOWED_FEATURES,
    "adapter_dir": str(ADAPTER_DIR),
    "metrics_path": str(eval_metrics_path),
}

slm_config_path = OUTPUT_DIR / "slm_query_parser_config.json"

with open(slm_config_path, "w", encoding="utf-8") as f:
    json.dump(slm_config, f, indent=2, ensure_ascii=False)

print("Saved config:", slm_config_path)
slm_config

Saved config: /content/drive/MyDrive/tablewise/artifacts_new/slm_query_parser_2/slm_query_parser_config.json


{'base_model': 'Qwen/Qwen2.5-1.5B-Instruct',
 'task': 'restaurant_query_parsing_to_json',
 'training_method': 'QLoRA / LoRA adapter fine-tuning',
 'num_train_examples': 8000,
 'num_validation_examples': 1000,
 'num_test_examples': 1000,
 'max_seq_length': 512,
 'json_schema_keys': ['city',
  'country',
  'price_bucket',
  'tags',
  'dietary',
  'matched_meals',
  'matched_features'],
 'allowed_price_buckets': ['cheap', 'mid', 'expensive'],
 'allowed_tags': ['italian',
  'french',
  'spanish',
  'greek',
  'portuguese',
  'german',
  'seafood',
  'steakhouse',
  'asian',
  'indian',
  'mexican',
  'mediterranean',
  'fast food',
  'coffee',
  'bar',
  'fine dining',
  'tapas'],
 'allowed_dietary': ['vegetarian friendly',
  'vegan options',
  'gluten free options'],
 'allowed_meals': ['breakfast', 'brunch', 'lunch', 'dinner', 'drinks'],
 'allowed_features': ['outdoor seating',
  'reservations',
  'delivery',
  'wheelchair accessible',
  'family friendly',
  'romantic'],
 'adapter_dir': '